# TACO and NEVO in one triple store

This notebook follows NICE-Food's `use_case_nice_food.ipynb` (rivm-syso/nicekg_analysis): the queries are `.rq` files, SPARQLWrapper sends them to a local Apache Jena Fuseki, pandas holds the results and networkx draws the graph.

Before running it, start Fuseki and load the TACO graph, the NEVO subgraphs of NICE-Food and the pinned FoodOn release, as `triplestore/README.md` describes:

```
python scripts/load_triplestore.py --foodon --tbca
```

Do not commit this notebook with its outputs: they hold NEVO values, which NEVO's conditions of use do not let this project redistribute (`data/SOURCES.md`).

In [ ]:
# import required packages
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from comparison_helper import agreement, identical_share, paired_agreement, values_by_food
from endpoint_helper import FUSEKI, bind, open_query, sparql_query_to_dataframe, substitute
from graph_helper import draw_food_graph, edges_to_graph

In [ ]:
endpoint = FUSEKI
OBO = "http://purl.obolibrary.org/obo/"
PROTEIN, CARBOHYDRATE = f"<{OBO}CDNO_0200040>", f"<{OBO}CDNO_0200005>"
same_class = open_query("../queries/nevo/taco_nevo_same_foodon_class.rq")
available_carbohydrate = open_query("../queries/nevo/taco_nevo_available_carbohydrate.rq")
food_graph = open_query("../queries/nevo/taco_nevo_food_graph.rq")

## The graph around one FoodOn class

Both tables follow the NICE-Food pattern: a food is typed with a FoodOn class, a nutrient value with a CDNO class. The query returns the edges around one FoodOn class: its superclasses, the TACO and NEVO foods typed with it, and their values for one CDNO class. Raw potato, with protein:

In [ ]:
edges = sparql_query_to_dataframe(
    substitute(food_graph, foodon=f"<{OBO}FOODON_03301449>", **{"class": PROTEIN}), endpoint=endpoint)
fig, ax = plt.subplots(figsize=(12, 7))
draw_food_graph(edges_to_graph(edges), ax=ax)
plt.show()

Chocolate milk, with CDNO's general carbohydrate class. TACO's carbohydrate is total carbohydrate by difference, fibre included; NICE-Food gives NEVO's *available* carbohydrate the same class. The graph also shows a NEVO chocolate bar typed as the drink: its value is far from the drinks'.

In [ ]:
edges = sparql_query_to_dataframe(
    substitute(food_graph, foodon=f"<{OBO}FOODON_03301816>", **{"class": CARBOHYDRATE}), endpoint=endpoint)
fig, ax = plt.subplots(figsize=(12, 7))
draw_food_graph(edges_to_graph(edges), ax=ax)
plt.show()

## Protein of foods with the same FoodOn class

The query pairs a TACO food and a NEVO food typed with the same FoodOn class, and their values typed with the same CDNO class in the same unit. Here the CDNO class is fixed to protein.

In [ ]:
protein = sparql_query_to_dataframe(bind(same_class, **{"class": PROTEIN}), endpoint=endpoint)
protein[["taco_value", "nevo_value"]] = protein[["taco_value", "nevo_value"]].astype(float)
print(len(protein), "pairs")
protein[protein.foodon.str.endswith("FOODON_03301449")][["taco_name", "taco_value", "nevo_name", "nevo_value"]]

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
sns.scatterplot(data=protein, x="nevo_value", y="taco_value", ax=ax)
top = max(protein.taco_value.max(), protein.nevo_value.max()) * 1.05
ax.plot([0, top], [0, top], color="grey", linestyle="--", linewidth=1)
ax.set(xlabel="NEVO protein (g per 100 g)", ylabel="TACO protein (g per 100 g)", xlim=(0, top), ylim=(0, top))
plt.show()

## Every CDNO class in one query

With rdflib the comparison has to run once per CDNO class; Fuseki answers the whole query at once. NEVO's available carbohydrate has its own query, which compares it with TACO's carbohydrate minus dietary fibre.

In [ ]:
pairs = sparql_query_to_dataframe(same_class, endpoint=endpoint)
carbohydrate = sparql_query_to_dataframe(available_carbohydrate, endpoint=endpoint)
print(len(pairs) + len(carbohydrate), "value pairs")
pairs.groupby("cdno_label").size().sort_values(ascending=False).head(15)

## More pairs through the FoodOn hierarchy

Only identical FoodOn classes pair foods above. `taco_nevo_related_foodon_class.rq` also pairs foods whose classes are one step apart in FoodOn's hierarchy, so that one is a more specific kind of the other. It needs the FoodOn release in the store (`--foodon`): the TACO graph carries only the classes TACO uses. Siblings under a common superclass are not paired, because FoodOn's superclasses include facets such as *food (raw)*.

In [ ]:
related = sparql_query_to_dataframe(open_query("../queries/nevo/taco_nevo_related_foodon_class.rq"), endpoint=endpoint)
same_foods = sparql_query_to_dataframe(open_query("../queries/nevo/taco_nevo_same_class_foods.rq"), endpoint=endpoint)
new_foods = set(related.taco_food) - set(same_foods.taco_food)
print(len(related), "food pairs;", len(new_foods), "TACO foods get their first NEVO food")
related.groupby(["relation", "taco_class_label", "nevo_class_label"]).size().sort_values(ascending=False).head(10)

Do these pairs compare as well as foods of the same class? Protein, each TACO food against the median of its NEVO foods:

In [ ]:
values = sparql_query_to_dataframe(
    substitute(open_query("../queries/nevo/values_in_cdno_class.rq"), **{"class": PROTEIN}), endpoint=endpoint)
food_pairs = pd.concat([same_foods, related[["relation", "taco_food", "nevo_food"]]])
agreement(food_pairs, values_by_food(values, "TACO"), values_by_food(values, "NEVO"))

## TBCA, a second Brazilian table (provisional)

TBCA, the Brazilian table of USP/FoRC, is compiled from many sources, as NEVO is, and was updated in 2025; TACO holds only its own analyses, made between 1999 and 2008. Comparing TBCA with NEVO and TACO with TBCA separates the country from the method. TBCA is used with its owner's permission and stays local (`data/SOURCES.md`); `scripts/fetch_tbca.py`, `scripts/tbca_foodon_candidates.py` and `scripts/tbca_to_rdf.py --candidates` prepare it.

TBCA's foods have no reviewed FoodOn type yet. They are linked through *candidates* matched from their English names (`tbca:foodOnCandidate`, never `rdf:type`), so every number below is provisional until the candidates are reviewed.

In [ ]:
tbca_nevo = open_query("../queries/tbca/tbca_nevo_candidate_class.rq")
taco_tbca = open_query("../queries/tbca/taco_tbca_candidate_class.rq")
AVAILABLE = f"<{OBO}CDNO_0200033>"
rows = []
for nutrient, cls in (("protein", PROTEIN), ("available carbohydrate", AVAILABLE)):
    pairs = sparql_query_to_dataframe(substitute(tbca_nevo, **{"class": cls}), endpoint=endpoint)
    rows.append({"comparison": "TBCA x NEVO", "nutrient": nutrient,
                 **paired_agreement(pairs, "tbca_food", "tbca_value", "nevo_value")})
pairs = sparql_query_to_dataframe(substitute(taco_tbca, **{"class": PROTEIN}), endpoint=endpoint)
rows.append({"comparison": "TACO x TBCA", "nutrient": "protein",
             **paired_agreement(pairs, "taco_food", "taco_value", "tbca_value")})
rows.append({"comparison": "TACO x NEVO", "nutrient": "protein",
             **paired_agreement(protein, "taco_food", "taco_value", "nevo_value")})
pd.DataFrame(rows)

Part of TBCA's values are TACO's: TBCA compiles other tables, TACO among them. Pairs with the same value are not independent measurements and should be set apart before reading TACO x TBCA as an effect of the method:

In [ ]:
same_values = {}
for nutrient, cls in (("protein", PROTEIN), ("water", f"<{OBO}CDNO_0200002>"), ("calcium", f"<{OBO}CDNO_0200544>"),
                      ("iron", f"<{OBO}CDNO_0200651>")):
    pairs = sparql_query_to_dataframe(substitute(taco_tbca, **{"class": cls}), endpoint=endpoint)
    same_values[nutrient] = f"{identical_share(pairs, 'taco_value', 'tbca_value'):.0%} of {len(pairs)}"
pd.Series(same_values, name="TACO and TBCA values within 0.5%")